In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


cwd = Path.cwd()
print(cwd)
root=cwd.parents[2]
pd.set_option("display.max_columns",None)
display(root)

c:\Users\sebas\PycharmProjects\Git\Seb_branch\institutional-roi-analysis\notebooks\04_modeling\02_national_models


WindowsPath('c:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

### Feature Selection for Explanatory Model

Variables used in the initial prediction model (e.g., credential level, distance, and other structural constraints) were excluded from the explanatory model.

This is because the residuals already represent performance after controlling for these factors. Including them again would introduce circular reasoning and reduce the interpretability of the results.

Instead, the explanatory model focuses on institutional characteristics and program composition variables that were not used in the prediction stage, allowing us to better understand what drives over- and underperformance.

In [2]:
driver_df=pd.read_csv(root/'data'/'raw'/'scorecard'/'raw_national_inst_driver.csv')

In [3]:
driver_df["has_endowment"] = (
    driver_df["endowment_begin"].notna() &
    driver_df["endowment_end"].notna()
).astype(int)

In [4]:
driver_df["has_endowment"].describe()

count    6322.000000
mean        0.422018
std         0.493920
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000
Name: has_endowment, dtype: float64

In [5]:
driver_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6322 entries, 0 to 6321
Data columns (total 49 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   program_percentage_agriculture                           5582 non-null   float64
 1   program_percentage_resources                             5582 non-null   float64
 2   program_percentage_architecture                          5582 non-null   float64
 3   program_percentage_ethnic_cultural_gender                5582 non-null   float64
 4   program_percentage_communication                         5582 non-null   float64
 5   program_percentage_communications_technology             5582 non-null   float64
 6   program_percentage_computer                              5582 non-null   float64
 7   program_percentage_personal_culinary                     5582 non-null   float64
 8   program_percentage_education

In [6]:
ridge_residual_df = pd.read_csv(root/"data"/"raw"/"scorecard"/"raw_ridge_national_residual_programs.csv")
xgb_residual_df = pd.read_csv(root/"data"/"raw"/"scorecard"/"raw_xgb_national_residual_programs.csv")

In [7]:
ridge_residual_df['sign_agreement'] = (
    np.sign(ridge_residual_df['1_year_error_log']) == 
    np.sign(ridge_residual_df['4_year_error_log'])
) & (
    np.sign(ridge_residual_df['4_year_error_log']) == 
    np.sign(ridge_residual_df['5_year_error_log'])
)

print(ridge_residual_df['sign_agreement'].value_counts(normalize=True))

print(ridge_residual_df[['1_year_error','4_year_error','5_year_error']].corr())

sign_agreement
True     0.602308
False    0.397692
Name: proportion, dtype: float64
              1_year_error  4_year_error  5_year_error
1_year_error      1.000000      0.786356      0.695285
4_year_error      0.786356      1.000000      0.799308
5_year_error      0.695285      0.799308      1.000000


In [8]:
xgb_residual_df['sign_agreement'] = (
    np.sign(xgb_residual_df['1_year_error']) == 
    np.sign(xgb_residual_df['4_year_error'])
) & (
    np.sign(xgb_residual_df['4_year_error']) == 
    np.sign(xgb_residual_df['5_year_error'])
)

print(xgb_residual_df['sign_agreement'].value_counts(normalize=True))

print(xgb_residual_df[['1_year_error','4_year_error','5_year_error']].corr())

sign_agreement
True     0.520556
False    0.479444
Name: proportion, dtype: float64
              1_year_error  4_year_error  5_year_error
1_year_error      1.000000      0.684768      0.570242
4_year_error      0.684768      1.000000      0.663753
5_year_error      0.570242      0.663753      1.000000


In [9]:
ridge_residual_df['average_error_log'] = ridge_residual_df[
    ["1_year_error_log", "4_year_error_log", "5_year_error_log"]
].median(axis=1)

In [10]:
xgb_residual_df['median_error_log'] = xgb_residual_df[
    ["1_year_error_log", "4_year_error_log", "5_year_error_log"]
].median(axis=1)

xgb_residual_df['total_pred'] = xgb_residual_df[
    ["1_year_pred", "4_year_pred", "5_year_pred"]
].sum(axis=1)

xgb_residual_df["median_pred_log"] = xgb_residual_df[
    ["1_year_pred_log", "4_year_pred_log", "5_year_pred_log"]
].median(axis=1)

xgb_residual_df["total_count"] = (
    xgb_residual_df["1_yr_working_count"] +
    xgb_residual_df["4_yr_working_count"] +
    xgb_residual_df["5_yr_working_count"]
)

k = np.percentile(np.log1p(xgb_residual_df["total_count"]), 75)

xgb_residual_df["weight"] = (
    np.log1p(xgb_residual_df["total_count"]) /
    np.log1p(xgb_residual_df["total_count"] + k)
)

In [11]:
print("XGBoost residual std:", xgb_residual_df["median_error_log"].std())
print("Ridge residual std:  ", ridge_residual_df["average_error_log"].std())

XGBoost residual std: 0.12981901211062838
Ridge residual std:   0.1603396555596885


In [12]:
join_cols=["unit_id"]
for col in join_cols:
    driver_df[col] = driver_df[col].astype(str).str.strip()
    xgb_residual_df[col] = xgb_residual_df[col].astype(str).str.strip()


### Target Variable Selection

While a composite scoring metric was developed to rank program-level variability, it was not used as the target for the explanatory model.

Instead, the model uses the average  raw percentage error of years 1, 4, and 5 as the target:

  * pct_error = error / predicted

This decision ensures that the model learns directly from observed over- and underperformance, rather than from a derived metric that incorporates additional adjustments (e.g., sample size penalties and variability scaling).

The composite score remains useful for identifying high-variability groups, but the explanatory model focuses on the underlying performance signal.

In [13]:
# residual_df["combined_pct_error"]=(
#     residual_df[['1_year_error','4_year_error','5_year_error']].sum(axis=1)
#     /
#     residual_df[['1_year_pred','4_year_pred','5_year_pred']].sum(axis=1)
# )

In [14]:
# ridge_residual_df["log_error_median"] = ridge_residual_df[
#     ["1_year_error", "4_year_error", "5_year_error"]
# ].median(axis=1)

# xgb_residual_df["log_pred_median"] = xgb_residual_df[
#     ["1_year_pred_log", "4_year_pred_log", "5_year_pred_log"]
# ].median(axis=1)

# ridge_residual_df["row_pct_error"] = (
#    ridge_residual_df["dollar_error_median"] / ridge_residual_df["dollar_pred_median"]
# )

In [15]:
# school_df = xgb_residual_df.groupby("unit_id", as_index=False).agg(
#     average_error_log  =("average_error_log", "median"),
#     average_pred_log =('log_pred_median', 'median'),
#     avg_rank_stability=("mean_rank_std_pct", "mean"),
#     school_name=("school_name", "first"),
#     total_count_1=("1_yr_working_count", "sum"),
#     total_count_4=("4_yr_working_count", "sum"),
#     total_count_5=("5_yr_working_count", "sum"),
# )

# school_df["total_count"] = (
#     school_df["total_count_1"] +
#     school_df["total_count_4"] +
#     school_df["total_count_5"]
# )

# k = np.percentile(np.log1p(school_df["total_count"]), 75)
# school_df["weight"] = (
#     np.log1p(school_df["total_count"]) /
#     np.log1p(school_df["total_count"] + k)
# )

In [16]:
xgb_residual_df=xgb_residual_df.drop(columns='school_name')

In [17]:
targ="median_error_log"
merge_df=driver_df.merge(xgb_residual_df, on=["unit_id"],how="inner")
merge_df.head()

,program_percentage_agriculture,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,school_name,unit_id,has_endowment,code,credential_level,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,5_year_error_log,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_year_error_log,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_year_error_log,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score,sign_agreement,median_error_log,total_pred,median_pred_log,total_count,weight
0,0.0407,0.0,0.0136,0.0,0.0,0.0542,0.0424,0.0,0.0424,0.1085,0.0203,0.0,0.0186,0.0,0.0119,0.0661,0.0,0.1424,0.0051,0.0,0.0,0.0373,0.0,0.0,0.0237,0.0,0.0559,0.0644,0.0441,0.022,0.0,0.0,0.0,0.0,0.0186,0.0,0.1678,0.0,7254.0,8699.0,0.6439,NaN,19.0,NaN,NaN,0.0,Alabama A & M University,100654,0,1101,3,1,Public,27.0,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.352968,11.003503,"Computer and Information Sciences, General.",0.349465,85218.0,60084.242,25133.757812,88490.0,11.390645,62110.523,11.036671,26379.476562,0.353974,39,63900.0,11.065075,43693.000,10.684943,20207.00000,0.380131,29.0,351,NaN,0.462477,0.443214,0.424718,0.411877,0.418309,0.399726,13.0,10.0,16.0,-3.0,6.0,3.0,3.000000,0.006818,0.117768,0.411877,True,0.353974,165887.765,11.003503,95.0,0.987154
1,0.0407,0.0,0.0136,0.0,0.0,0.0542,0.0424,0.0,0.0424,0.1085,0.0203,0.0,0.0186,0.0,0.0119,0.0661,0.0,0.1424,0.0051,0.0,0.0,0.0373,0.0,0.0,0.0237,0.0,0.0559,0.0644,0.0441,0.022,0.0,0.0,0.0,0.0,0.0186,0.0,0.1678,0.0,7254.0,8699.0,0.6439,NaN,19.0,NaN,NaN,0.0,Alabama A & M University,100654,0,1312,5,2,Public,18.0,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.142760,10.903197,Teacher Education and Professional Development...,0.239563,69062.0,54349.855,14712.144531,60412.0,11.008943,53289.234,10.883490,7122.765625,0.125453,24,56295.0,10.938361,49727.000,10.814303,6568.00000,0.124058,21.0,283,NaN,0.132081,0.124184,0.133662,0.126653,0.270693,0.251821,29

In [18]:
model_df=merge_df.copy()
print("scorecard driver schools:", driver_df['unit_id'].nunique())
print("xgb_residual_df school:", xgb_residual_df['unit_id'].nunique())
print("model_df schools:", model_df['unit_id'].nunique())

print("model_df columns:")
print(sorted(model_df.columns.tolist()))

scorecard driver schools: 6322
xgb_residual_df school: 4327
model_df schools: 4327
model_df columns:
['1_year_earning', '1_year_earning_log', '1_year_error', '1_year_error_log', '1_year_pct_error', '1_year_pred', '1_year_pred_log', '1_year_score', '1_yr_working_count', '4_year_earning', '4_year_earning_log', '4_year_error', '4_year_error_log', '4_year_pct_error', '4_year_pred', '4_year_pred_log', '4_year_score', '4_yr_working_count', '5_year_earning', '5_year_earning_log', '5_year_error', '5_year_error_log', '5_year_pct_error', '5_year_pred', '5_year_pred_log', '5_year_score', '5_yr_working_count', 'act_scores_midpoint_cumulative', 'admission_rate_overall', 'age_entry', 'carnegie_size_setting', 'code', 'confidence', 'credential_level', 'distance', 'dolflag', 'endowment_begin', 'endowment_end', 'faculty_salary', 'ft_faculty_rate', 'grad_students', 'has_endowment', 'instructional_expenditure_per_fte', 'locale', 'location_lat', 'location_lon', 'mean_rank_std_pct', 'median_error_log', 'med

In [19]:
model_df=model_df.drop(columns=[
    # "program_reporter_programs_offered",
    # 'code',
    # 'unit_id',
    # 'school_name_y'
    ]
)

In [20]:
program_cols = [c for c in model_df.columns if c.startswith("program_percentage_")]

for c in program_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0)

# STEM
model_df["pct_stem"] = model_df[
    [
        "program_percentage_computer",
        "program_percentage_engineering",
        "program_percentage_engineering_technology",
        "program_percentage_mathematics",
        "program_percentage_physical_science",
        "program_percentage_biological",
        "program_percentage_science_technology"
    ]
].sum(axis=1)

# Business / Econ
model_df["pct_business"] = model_df[
    ["program_percentage_business_marketing"]
].sum(axis=1)

# Health
model_df["pct_health"] = model_df[
    ["program_percentage_health"]
].sum(axis=1)

# Social Sciences
model_df["pct_social_science"] = model_df[
    [
        "program_percentage_psychology",
        "program_percentage_social_science",
        "program_percentage_history",
        "program_percentage_public_administration_social_service"
    ]
].sum(axis=1)

# Humanities
model_df["pct_humanities"] = model_df[
    [
        "program_percentage_english",
        "program_percentage_language",
        "program_percentage_humanities",
        "program_percentage_philosophy_religious",
        "program_percentage_theology_religious_vocation",
        "program_percentage_ethnic_cultural_gender"
    ]
].sum(axis=1)

# Arts & Communication
model_df["pct_arts_comm"] = model_df[
    [
        "program_percentage_visual_performing",
        "program_percentage_communication"
    ]
].sum(axis=1)

# Education
model_df["pct_education"] = model_df[
    ["program_percentage_education"]
].sum(axis=1)

# Trades / Technical
model_df["pct_trades"] = model_df[
    [
        "program_percentage_construction",
        "program_percentage_mechanic_repair_technology",
        "program_percentage_precision_production",
        "program_percentage_transportation"
    ]
].sum(axis=1)

# Services / Consumer
model_df["pct_services"] = model_df[
    [
        "program_percentage_personal_culinary",
        "program_percentage_family_consumer_science",
        "program_percentage_parks_recreation_fitness"
    ]
].sum(axis=1)

# Law / Security
model_df["pct_law_security"] = model_df[
    [
        "program_percentage_legal",
        "program_percentage_security_law_enforcement"
    ]
].sum(axis=1)

# Agriculture / Natural resources
model_df["pct_agriculture"] = model_df[
    [
        # "program_percentage_agriculture",
        "program_percentage_resources"
    ]
].sum(axis=1)

model_df["pct_high_roi"] = (
    model_df["program_percentage_engineering"] +
    model_df["program_percentage_computer"] +
    model_df["program_percentage_health"]
)

model_df["pct_low_roi"] = (
    model_df["program_percentage_education"] +
    model_df["program_percentage_personal_culinary"] +
    model_df["program_percentage_humanities"]
)

model_df["program_hhi"] = (model_df[program_cols] ** 2).sum(axis=1)

model_df["max_program_share"] = model_df[program_cols].max(axis=1)

model_df["high_roi_x_concentration"] = (
    model_df["pct_high_roi"] * model_df["program_hhi"]
)

# model_df = model_df.drop(columns=program_cols)



In [21]:
ignore_cols =['school_name', '5_year_error_log',
       '5_year_earning', '5_year_pred', '5_year_error', '4_year_earning',
       '4_year_earning_log', '4_year_pred', '4_year_pred_log', '4_year_error',
       '4_year_error_log', '4_yr_working_count', '1_year_earning',
       '1_year_earning_log', '1_year_pred', '1_year_pred_log', '1_year_error',
       '1_year_error_log', '1_yr_working_count', 'school_count', 'confidence',
       '1_year_pct_error', '1_year_score', '4_year_pct_error', '4_year_score',
       '5_year_pct_error', '5_year_score', 'rank_1', 'rank_4', 'rank_5',
       'move_1_to_4', 'move_4_to_5', 'move_1_to_5', 'rank_std', 'rank_std_pct',
       'mean_rank_std_pct', 'median_score', 'sign_agreement',
       'log_error_median', 'log_pred_median','5_year_earning_log',
       '5_year_pred_log','5_yr_working_count','weight' ,'median_pred_log','total_pred',
]

In [22]:
display(model_df.describe())

numeric_cols = model_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ignore_cols]
display("Feature correlation to target variable",
        model_df[numeric_cols].corr()[targ].sort_values())
display(model_df.info())

,program_percentage_agriculture,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,has_endowment,code,credential_level,distance,5_yr_working_count,location_lat,location_lon,locale,admission_rate_overall,median_family_income,students_with_pell_grant,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,5_year_earning_log,5_year_pred_log,5_year_error_log,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_year_error_log,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_year_error_log,1_yr_working_count,school_count,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score,median_error_log,total_pred,median_pred_log,total_count,weight,pct_stem,pct_business,pct_health,pct_social_science,pct_humanities,pct_arts_comm,pct_education,pct_trades,pct_services,pct_law_security,pct_agriculture,pct_high_roi,pct_low_roi,program_hhi,max_program_share,high_roi_x_concentration
count,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36211.000000,34043.000000,33424.000000,2959.000000,35972.000000,2.906500e+04,2.906500e+04,35529.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,25209.000000,36044.000000,35156.000000,36044.000000,36219.000000,19238.000000,17249.000000,26437.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,36219.000000,362

'Feature correlation to target variable'

program_percentage_visual_performing             -0.051474
pct_arts_comm                                    -0.043436
program_percentage_theology_religious_vocation   -0.034544
program_percentage_education                     -0.030511
pct_education                                    -0.030511
                                                    ...   
program_percentage_engineering                    0.028971
program_percentage_computer                       0.029364
pct_stem                                          0.031961
faculty_salary                                    0.050419
median_error_log                                  1.000000
Name: median_error_log, Length: 79, dtype: float64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36219 entries, 0 to 36218
Columns: 130 entries, program_percentage_agriculture to high_roi_x_concentration
dtypes: bool(1), float64(112), int64(8), object(9)
memory usage: 35.7+ MB


None

In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer

def run_school_corr_screen(
    merge_df,
    feature_cols,
    target_col="median_error_log",
    weight_col="weight"
):
    # 1. Remove duplicate column names
    df_clean = merge_df.loc[:, ~merge_df.columns.duplicated()].copy()

    # 2. Build a safe feature list
    clean_feature_cols = [
        c for c in feature_cols
        if c in df_clean.columns and c not in [target_col, weight_col]
    ]

    cols_to_keep = clean_feature_cols + [target_col, weight_col]
    df_tmp = df_clean[cols_to_keep].copy()

    # 3. Drop missing target and rows with all-feature missing
    df_tmp = df_tmp.dropna(subset=[target_col])
    df_tmp = df_tmp.dropna(subset=clean_feature_cols, how="all")

    print(f"Rows after dropna: {len(df_tmp)}")
    print(f"Features used: {len(clean_feature_cols)}")

    # 4. Clip target
    target_series = df_tmp[target_col]

    if isinstance(target_series, pd.DataFrame):
        raise ValueError(
            f"target_col resolved to multiple columns: {target_series.columns.tolist()}"
        )

    lo = float(target_series.quantile(0.02))
    hi = float(target_series.quantile(0.98))

    print(f"Target distribution (Clipped at {lo:.4f}, {hi:.4f}):")
    print(f"{target_series.describe().round(4)}\n")

    df_tmp[target_col] = df_tmp[target_col].clip(lower=lo, upper=hi)

    # 5. Prepare X, y, w
    X = df_tmp[clean_feature_cols].apply(pd.to_numeric, errors="coerce")

    all_nan_cols = X.columns[X.isna().all()].tolist()
    if all_nan_cols:
        print(f"\nDropping {len(all_nan_cols)} non-numeric/all-NaN columns before imputation:")
        print(all_nan_cols)

    X = X.drop(columns=all_nan_cols)

    used_feature_cols = X.columns.tolist()
    print(f"Features after dropping unusable cols: {len(used_feature_cols)}")
    
    y = df_tmp[target_col]
    w = df_tmp[weight_col]

    # 6. Correlation screen
    corr = X.corrwith(y).abs().sort_values(ascending=False)

    print("Top 15 correlations with target:")
    print(corr.head(15).round(4).to_string())
    print(f"\nFeatures with |corr| > 0.10: {(corr > 0.10).sum()}")
    print(f"Features with |corr| > 0.15: {(corr > 0.15).sum()}")
    print(f"Features with |corr| > 0.20: {(corr > 0.20).sum()}")

    # 7. Impute
    imputer = SimpleImputer(strategy="median")
    X_imp_arr = imputer.fit_transform(X)
    X_imp = pd.DataFrame(X_imp_arr, columns=X.columns, index=X.index)

    # 8. Cross-validated random forest
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=3,
        random_state=42,
        n_jobs=-1
    )

    fold_r2s = []

    for train_idx, val_idx in kf.split(X_imp):
        X_tr, X_val = X_imp.iloc[train_idx], X_imp.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        w_tr, w_val = w.iloc[train_idx], w.iloc[val_idx]

        rf.fit(X_tr, y_tr, sample_weight=w_tr)
        y_pred = rf.predict(X_val)
        fold_r2s.append(r2_score(y_val, y_pred, sample_weight=w_val))

    print(f"\nRF CV R² (max_depth=3): {np.mean(fold_r2s):.4f} ± {np.std(fold_r2s):.4f}")
    print(f"Fold R²s: {[round(s, 3) for s in fold_r2s]}")

    # 9. Feature importances
    rf.fit(X_imp, y, sample_weight=w)
    imp_df = pd.DataFrame({
        "feature": X.columns,
        "importance": rf.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\nTop 10 feature importances:")
    print(imp_df.head(10).to_string(index=False))

    return corr, imp_df

model_df = model_df.loc[:, ~model_df.columns.duplicated()].copy()
feature_cols = [c for c in model_df.columns if c not in ignore_cols]
corr_results, imp_results = run_school_corr_screen(model_df, feature_cols)

Rows after dropna: 36219
Features used: 85
Target distribution (Clipped at -0.2662, 0.2976):
count    36219.0000
mean         0.0045
std          0.1298
min         -0.8868
25%         -0.0672
50%          0.0029
75%          0.0712
max          1.0901
Name: median_error_log, dtype: float64


Dropping 4 non-numeric/all-NaN columns before imputation:
['school_type', 'school_state', 'selectivity_bucket', 'title']
Features after dropping unusable cols: 81
Top 15 correlations with target:
faculty_salary                                    0.0496
program_percentage_visual_performing              0.0492
pct_arts_comm                                     0.0411
program_percentage_theology_religious_vocation    0.0350
pct_education                                     0.0332
program_percentage_education                      0.0332
program_percentage_computer                       0.0312
pct_stem                                          0.0307
program_percentage_engineering                    0.03

In [24]:
import plotly.express as px

pre_y = model_df[targ]

percentiles = [0.1, 0.2, 0.3, 0.7, 0.8, 0.9]
values = np.quantile(pre_y, percentiles)

fig = px.histogram(pre_y, nbins=100, title="Target Distribution with Percentiles")

for p, v in zip(percentiles, values):
    fig.add_vline(
        x=v,
        line_dash="dash",
        annotation_text=f"{int(p*100)}%",
        annotation_position="top"
    )

fig.show()

In [25]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, PolynomialFeatures
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
RANDOM_STATE = 42

In [26]:
feature_cols = [c for c in model_df.columns if c not in ignore_cols + [targ, "weight",'title','unit_id','code']]

# y_raw = np.log1p(model_df[targ])
# y_raw = y.clip(upper=y.quantile(0.95))
y_raw = model_df[targ].copy()

low = y_raw.quantile(0.3)
high = y_raw.quantile(0.7)

mask = (y_raw <= low) | (y_raw >= high)

X_clf = model_df[feature_cols].loc[mask].copy()
y_clf = (y_raw.loc[mask] >= high).astype(int)
w_clf = model_df['weight'].loc[mask].copy()

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_clf,
    y_clf,
    w_clf,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y_clf
)


num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

In [28]:
pipe_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        solver="lbfgs",
        C=0.5
    ))
])

pipe_lr.fit(X_train, y_train, clf__sample_weight=w_train)

,steps,"[('preprocessor', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [29]:
preds = pipe_lr.predict(X_test)
probs = pipe_lr.predict_proba(X_test)[:, 1]

print(classification_report(y_test, preds))
print("AUC:", roc_auc_score(y_test, probs, sample_weight=w_test))

              precision    recall  f1-score   support

           0       0.59      0.57      0.58      3260
           1       0.59      0.61      0.60      3260

    accuracy                           0.59      6520
   macro avg       0.59      0.59      0.59      6520
weighted avg       0.59      0.59      0.59      6520

AUC: 0.622122556789344


In [30]:
feature_names = pipe_lr.named_steps["preprocessor"].get_feature_names_out()
coefs = pipe_lr.named_steps["clf"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs
}).sort_values("coef", ascending=False)

print(coef_df.head(20))
print(coef_df.tail(20))

                             feature      coef
121             cat__school_state_PR  0.956958
127             cat__school_state_UT  0.595212
83              cat__school_state_AZ  0.573939
89              cat__school_state_FL  0.442059
118             cat__school_state_OK  0.438074
95              cat__school_state_IL  0.422214
144  cat__carnegie_size_setting_18.0  0.419042
104             cat__school_state_MN  0.416129
90              cat__school_state_GA  0.411782
135   cat__carnegie_size_setting_1.0  0.387683
101             cat__school_state_MD  0.382324
110             cat__school_state_ND  0.346690
39               num__faculty_salary  0.308242
126             cat__school_state_TX  0.286986
131             cat__school_state_WA  0.262551
148   cat__carnegie_size_setting_5.0  0.254095
75            num__max_program_share  0.192375
128             cat__school_state_VA  0.174709
132             cat__school_state_WI  0.165332
103             cat__school_state_MI  0.163275
             

1. Even after controlling for institutional and program-level constraints, geographic location remains a strong predictor of over- and underperformance, suggesting regional effects are not fully captured by baseline models.
2. Programs at institutions with higher faculty compensation tend to exceed expected outcomes, suggesting resource quality may play a role beyond structural constraints
3. Institutions with concentrated program focus may outperform expectations, possibly due to specialization effects.

In [31]:
from xgboost import XGBClassifier

pipe_xgbc = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        base_score=0.5
    ))
])

In [32]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    "clf__n_estimators": randint(100, 600),
    "clf__learning_rate": uniform(0.02, 0.08),
    "clf__max_depth": randint(3, 7),
    "clf__subsample": uniform(0.6, 0.3),
    "clf__colsample_bytree": uniform(0.6, 0.3),
    "clf__min_child_weight": [0, 1, 4],
    "clf__gamma": uniform(0.0, 0.2),
}

search = RandomizedSearchCV(
    pipe_xgbc,
    param_distributions=param_dist,
    n_iter=40,
    scoring="roc_auc",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=3,
    verbose=1
)

search.fit(X_train, y_train, clf__sample_weight=w_train)

print("Best AUC:", search.best_score_)

print("XGB Best params:", search.best_params_)
print("XGB Best CV MAE:", round(-search.best_score_, 2))

best_xgb = search.best_estimator_
xgb_preds = search.predict(X_test)

print("XGB Test:", classification_report(y_test, xgb_preds, sample_weight=w_test))
print("XGB Test AUC:", roc_auc_score(y_test, xgb_preds, sample_weight=w_test))

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best AUC: 0.6685196369610129
XGB Best params: {'clf__colsample_bytree': np.float64(0.6265477506155759), 'clf__gamma': np.float64(0.039196572483829045), 'clf__learning_rate': np.float64(0.023618183112843045), 'clf__max_depth': 6, 'clf__min_child_weight': 0, 'clf__n_estimators': 307, 'clf__subsample': np.float64(0.8241960330412142)}
XGB Best CV MAE: -0.67
XGB Test:               precision    recall  f1-score   support

           0       0.63      0.61      0.62 3231.6026934591273
           1       0.62      0.64      0.63 3234.6376029067915

    accuracy                           0.63 6466.240296365919
   macro avg       0.63      0.63      0.63 6466.240296365919
weighted avg       0.63      0.63      0.63 6466.240296365919

XGB Test AUC: 0.6262363650850713


In [33]:
preds = search.predict(X_test)
probs = search.predict_proba(X_test)[:, 1]

print(classification_report(y_test, preds, sample_weight=w_test))
print("AUC:", roc_auc_score(y_test, probs, sample_weight=w_test))

preds = (probs >= 0.5).astype(int)

results = X_test.copy()
results["y_true"] = y_test
results["y_pred"] = preds
results["prob"] = probs
results["correct"] = (results["y_true"] == results["y_pred"])

              precision    recall  f1-score   support

           0       0.63      0.61      0.62 3231.6026934591273
           1       0.62      0.64      0.63 3234.6376029067915

    accuracy                           0.63 6466.240296365919
   macro avg       0.63      0.63      0.63 6466.240296365919
weighted avg       0.63      0.63      0.63 6466.240296365919

AUC: 0.6715073007902322


In [39]:
import plotly.graph_objects as go
from sklearn.metrics import log_loss, f1_score, precision_score, recall_score, accuracy_score, confusion_matrix

thresholds = np.linspace(0.01, 0.99, 200)

metrics = {
    "f1": [], "precision": [], "recall": [], "accuracy": []
}

for t in thresholds:
    preds = (probs >= t).astype(int)
    metrics["f1"].append(f1_score(y_test, preds, zero_division=0))
    metrics["precision"].append(precision_score(y_test, preds, zero_division=0))
    metrics["recall"].append(recall_score(y_test, preds, zero_division=0))
    metrics["accuracy"].append(accuracy_score(y_test, preds))

best_idx = np.argmax(metrics["f1"])
best_thresh = thresholds[best_idx]

fig = go.Figure()

fig.add_trace(go.Scatter(x=thresholds, y=metrics["f1"],
    name="F1", line=dict(color="#636EFA", width=2)))
fig.add_trace(go.Scatter(x=thresholds, y=metrics["precision"],
    name="Precision", line=dict(color="#EF553B", width=2)))
fig.add_trace(go.Scatter(x=thresholds, y=metrics["recall"],
    name="Recall", line=dict(color="#00CC96", width=2)))
fig.add_trace(go.Scatter(x=thresholds, y=metrics["accuracy"],
    name="Accuracy", line=dict(color="#FFA15A", width=2)))

fig.add_vline(
    x=best_thresh,
    line_dash="dash",
    line_color="white",
    annotation_text=f"Best F1 threshold: {best_thresh:.3f}",
    annotation_position="top right"
)

fig.update_layout(
    title="Threshold vs Classification Metrics (Train)",
    xaxis_title="Threshold",
    yaxis_title="Score",
    legend=dict(orientation="h", y=-0.15),
    template="plotly_dark",
    hovermode="x unified"
)

fig.show()
print(f"\nBest F1 threshold: {best_thresh:.4f}")
print(f"At this threshold — F1: {metrics['f1'][best_idx]:.4f} | "
      f"Precision: {metrics['precision'][best_idx]:.4f} | "
      f"Recall: {metrics['recall'][best_idx]:.4f}")


Best F1 threshold: 0.3744
At this threshold — F1: 0.6810 | Precision: 0.5534 | Recall: 0.8850


In [40]:
train_proba = search.predict_proba(X_train)[:, 1]
test_proba = search.predict_proba(X_test)[:, 1]

In [41]:
train_pred = (train_proba >= best_thresh).astype(int)
test_pred = (test_proba >= best_thresh).astype(int)

train_auc = roc_auc_score(y_train, train_proba, sample_weight=w_train)
test_auc = roc_auc_score(y_test, test_proba, sample_weight=w_test)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

train_logloss = log_loss(y_train, train_proba, sample_weight=w_train)
test_logloss = log_loss(y_test, test_proba, sample_weight=w_test)

print("\nHGBClassifier")
print("Train AUC:", round(train_auc, 4))
print("Test AUC:", round(test_auc, 4))
print("Train Accuracy:", round(train_acc, 4))
print("Test Accuracy:", round(test_acc, 4))
print("Train LogLoss:", round(train_logloss, 4))
print("Test LogLoss:", round(test_logloss, 4))

print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report (Test):")
print(classification_report(y_test, test_pred, digits=4))


HGBClassifier
Train AUC: 0.8238
Test AUC: 0.6715
Train Accuracy: 0.6655
Test Accuracy: 0.5854
Train LogLoss: 0.5624
Test LogLoss: 0.6467

Confusion Matrix (Test):
[[ 932 2328]
 [ 375 2885]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     0.7131    0.2859    0.4081      3260
           1     0.5534    0.8850    0.6810      3260

    accuracy                         0.5854      6520
   macro avg     0.6333    0.5854    0.5446      6520
weighted avg     0.6333    0.5854    0.5446      6520



In [42]:
over = results[results["y_true"] == 1]
under = results[results["y_true"] == 0]

In [43]:
for col in num_cols:
    over_mean = over[col].mean()
    under_mean = under[col].mean()
    std = results[col].std()

    if std == 0:
        continue

    effect = (over_mean - under_mean) / std

    if abs(effect) > 0.09:
        print(f"{col}: effect={effect:.2f}")

faculty_salary: effect=0.12


Higher faculty salaries are weakly associated with overperformance, suggesting that resource availability or faculty quality may contribute to outcomes beyond baseline expectations.

In [44]:
for col in cat_cols:
    print(f"\n==== {col} ====")

    over_dist = over[col].value_counts(normalize=True)
    under_dist = under[col].value_counts(normalize=True)

    combined = pd.concat([over_dist, under_dist], axis=1)
    combined.columns = ["over", "under"]
    combined["diff"] = combined["over"] - combined["under"]
    combined = combined[combined["diff"].abs() > 0.05]

    print(combined.sort_values("diff", ascending=False).head(10))


==== school_type ====
Empty DataFrame
Columns: [over, under, diff]
Index: []

==== school_state ====
Empty DataFrame
Columns: [over, under, diff]
Index: []

==== carnegie_size_setting ====
Empty DataFrame
Columns: [over, under, diff]
Index: []

==== open_admissions_policy ====
Empty DataFrame
Columns: [over, under, diff]
Index: []

==== selectivity_bucket ====
Empty DataFrame
Columns: [over, under, diff]
Index: []


No single variable strongly differentiates over- and underperformers; instead, results suggest that performance deviations arise from weak individual effects and more complex interactions between features.

In [45]:
feature_names = search.best_estimator_.named_steps["preprocessor"].get_feature_names_out()
importances = search.best_estimator_.named_steps["clf"].feature_importances_

imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print(imp_df.head(20))

                                       feature  importance
120                       cat__school_state_PA    0.014038
95                        cat__school_state_IL    0.012091
39                         num__faculty_salary    0.012022
98                        cat__school_state_KY    0.011285
59                          num__grad_students    0.011182
117                       cat__school_state_OH    0.010640
89                        cat__school_state_FL    0.010375
122                       cat__school_state_RI    0.010316
99                        cat__school_state_LA    0.009742
127                       cat__school_state_UT    0.009740
123                       cat__school_state_SC    0.009651
149             cat__carnegie_size_setting_6.0    0.009513
142            cat__carnegie_size_setting_16.0    0.009460
35              num__program_percentage_health    0.009448
86                        cat__school_state_CT    0.009434
34   num__program_percentage_visual_performing    0.0091

Residual behavior is not well explained by linear relationships alone; nonlinear interactions between institutional constraints significantly improve classification performance.

1. Higher-resource institutions tend to deviate positively from expectations, suggesting baseline models do not fully capture institutional quality.
2. Program composition influences over/underperformance, even after controlling for baseline expectations.

In [46]:
# Replace 'institution_type' with any feature you want to audit
for col in model_df.columns:
    if col not in ignore_cols:
        # Group and calculate metrics
        bias_audit = model_df.groupby(col)[targ].agg(['mean', 'std', 'count'])
        
        # Filter for groups with at least 10 schools to ignore 'noisy' outliers
        significant_bias = bias_audit[bias_audit['count'] >= 10].sort_values(['mean'])
        
        if not significant_bias.empty:
            print(f"\n--- Bias Audit for: {col} ---")
            # We look for means far from 0.0
            print(significant_bias)


--- Bias Audit for: program_percentage_agriculture ---
                                    mean       std  count
program_percentage_agriculture                           
0.1046                         -0.098153  0.093737     10
0.0061                         -0.097054  0.186282     24
0.0342                         -0.087275  0.253953     14
0.0232                         -0.082900  0.088355     15
0.1698                         -0.076349  0.093682     13
...                                  ...       ...    ...
0.0243                          0.093132  0.093731     16
0.0034                          0.093330  0.224758     15
0.1243                          0.104104  0.282277     19
0.0011                          0.119777  0.137338     19
0.0405                          0.121961  0.312269     10

[245 rows x 3 columns]

--- Bias Audit for: program_percentage_resources ---
                                  mean       std  count
program_percentage_resources                           


The bias audit reveals that while the model captures overall patterns well, residuals exhibit systematic structure across geographic regions, program composition, and institutional scale. Resource-related variables show non-linear effects, and no single feature fully explains deviations, suggesting that performance differences arise from complex interactions rather than isolated factors.